# A100 Final Run — Rebuilt Architecture

Trains the consolidated `a100_final` architecture (Sept 2026 rebuild): sigma fed into the direction-weighting pathway, a blur-deconvolved normalizer, blur-corrected higher-moment candidate vectors, a widened (K=10) von Mises-Fisher posterior, a blur-adaptive graph, epoch-based shuffled sampling, and a multiprocess data loader -- on the expanded 1,009,384-track corpus (24.9/30.0/45.2 low/mid/high energy weighting). Includes a small-vs-large capacity scaling ladder to finally answer whether the winner architecture was undersized.

**Crash-resilient by design.** Every checkpoint (every ~20 min) writes DIRECTLY to a Google-Drive-mounted results folder, and training auto-resumes from the last checkpoint on restart (`RESUME=1` is the default). If Colab disconnects mid-run -- a real possibility on a run this long, even on Pro/Pro+ -- just re-run the training cell for that config; it picks up exactly where it left off, no manual recovery needed. There is no separate "save to Drive" step at the end -- it's live throughout the whole run.

**Before you start -- upload the two data files to Google Drive**, folder `MyDrive/siimpl_rot/` (same convention as previous runs):
- `siimpl_train_v2.csv` (~7.9 GB) -- local path `C:\Inverse ML\data\siimpl_rot\siimpl_train_v2.csv`
- `siimpl_eval_v2.csv` (~0.9 GB) -- local path `C:\Inverse ML\data\siimpl_rot\siimpl_eval_v2.csv`

Everything else (code, checkpoints, eval results) is handled automatically by this notebook. Run the cells top to bottom in order.

In [ ]:
# 1. Confirm A100 80GB
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone the code (branch a100-final-rebuild)
#    If the repo is private, set a Colab Secret named GH_TOKEN (key icon in the left sidebar).
import os
GH_TOKEN = os.environ.get('GH_TOKEN', '')
try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or userdata.get('GH_TOKEN')
except Exception:
    pass
REPO = 'github.com/cbharathulwar/sbi-srim.git'
BRANCH = 'a100-final-rebuild'
url = f"https://{(GH_TOKEN + '@') if GH_TOKEN else ''}{REPO}"
%cd /content
!rm -rf sbi-srim
!git clone --branch $BRANCH --single-branch $url sbi-srim
%cd /content/sbi-srim
!git log --oneline -1
!pip -q install scipy 2>/dev/null; echo deps-ok

## 3. Mount Drive, stage data locally, point results at Drive

Data is copied to local Colab disk (`/content/sbi-srim/data/...`) for fast random-access reads during training -- this is a one-time ~1-2 min copy, not a per-step cost. `RESULTS_DIR` is left pointing at the Drive mount itself, since checkpoints are small, infrequent writes and this is exactly what gives crash resilience.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

DATA_SRC = '/content/drive/MyDrive/siimpl_rot'
DATA_DST = '/content/sbi-srim/data/siimpl_rot'
os.makedirs(DATA_DST, exist_ok=True)
for f in ['siimpl_train_v2.csv', 'siimpl_eval_v2.csv']:
    s = os.path.join(DATA_SRC, f)
    assert os.path.exists(s), (
        f'MISSING in Drive: {s}\n'
        f'-> Upload it to Google Drive at MyDrive/siimpl_rot/{f} first, then re-run this cell.'
    )
    d = os.path.join(DATA_DST, f)
    print(f'copying {f} ({os.path.getsize(s)/1e9:.2f} GB) ...')
    shutil.copy(s, d)
print('data staged.')
!ls -la /content/sbi-srim/data/siimpl_rot/

# RESULTS live DIRECTLY on Drive -- every ~20 min train.py writes checkpoint_latest.pt
# here. A Colab disconnect only loses at most ~20 min of progress; re-running the
# training cell resumes automatically (RESUME=1 is the default in config.py).
RESULTS_ROOT = '/content/drive/MyDrive/a100_final_results'
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.makedirs(f'{RESULTS_ROOT}/small', exist_ok=True)
os.makedirs(f'{RESULTS_ROOT}/large', exist_ok=True)
print('results will autosave to:', RESULTS_ROOT)

## 4. Estimate total training time (measured on THIS GPU, not guessed)

The only real throughput number available before this cell was measured on a 2080 Ti at a much smaller batch size, so it is not a reliable predictor of A100 performance -- a napkin-math extrapolation from it could plausibly be off by 2-3x in either direction. Rather than trust that, this cell runs a short (200-step), throwaway calibration pass for each config on the actual GPU you're using right now, reads the real measured `it/s`, and computes how many steps that config can actually complete inside its share of the 30-hour budget. This is the same approach `launch_a100.sh` uses (`TARGET_STEPS = it/s * budget_seconds * 0.92`, an 8% safety margin for periodic-eval overhead) -- reimplemented here as a notebook cell since the notebook drives `train.py` directly rather than the shell launcher.

Takes a few minutes total (200 steps x 2 configs). The recommended step counts below are what the training cells in sections 5 and 6 actually use.

In [ ]:
import os, re, subprocess, time

TOTAL_HOURS = 30.0   # overall budget across BOTH configs, sequential on one GPU
MARGIN_H = 1.0        # per-config margin for CSV load + periodic/final eval overhead
CALIB_STEPS = 200

PER_CONFIG_H = max(0.5, (TOTAL_HOURS - 2 * MARGIN_H) / 2)
print(f'[PLAN] total budget={TOTAL_HOURS:.1f}h -> per-config={PER_CONFIG_H:.2f}h '
      f'(small + large run back to back on one GPU)')


def calibrate(config_name):
    """Runs CALIB_STEPS throwaway steps for `config_name` on the real GPU and
    returns the measured it/s from train.py's own [THROUGHPUT] line."""
    calib_dir = f'/content/sbi-srim/_calib_{config_name}'
    env = os.environ.copy()
    env.update({
        'KMP_DUPLICATE_LIB_OK': 'TRUE', 'PYTHONIOENCODING': 'utf-8',
        'CONFIG': config_name, 'RESULTS_DIR': calib_dir, 'RESUME': '0',
        'TARGET_STEPS': str(CALIB_STEPS), 'TIME_BUDGET_HOURS': '0.5',
        'CHECKPOINT_EVERY_SEC': '1000000', 'EVAL_N_PER_BIN': '50',
        'TRAIN_CSV': 'data/siimpl_rot/siimpl_train_v2.csv',
        'EVAL_CSV': 'data/siimpl_rot/siimpl_eval_v2.csv',
    })
    print(f'[CALIBRATE] {config_name}: running {CALIB_STEPS} throwaway steps on the real GPU ...')
    t0 = time.time()
    result = subprocess.run(
        ['python', 'smearing_resolution/architecture_experiments/a100_final/train.py'],
        cwd='/content/sbi-srim', env=env, capture_output=True, text=True)
    out = result.stdout + result.stderr
    m = re.search(r'\[THROUGHPUT\]\s*([0-9.]+)\s*it/s', out)
    if not m:
        print(out[-4000:])
        raise RuntimeError(f'Could not parse [THROUGHPUT] for {config_name} -- see output above.')
    ips = float(m.group(1))
    print(f'[CALIBRATE] {config_name}: measured {ips:.3f} it/s '
          f'({time.time()-t0:.0f}s wall for the calibration pass itself)')
    return ips


def recommend_steps(config_name):
    ips = calibrate(config_name)
    steps = int(ips * PER_CONFIG_H * 3600 * 0.92)
    hours_for_calib_steps = CALIB_STEPS / ips / 3600
    print(f'[ESTIMATE] {config_name}: at {ips:.3f} it/s, the {PER_CONFIG_H:.2f}h budget '
          f'fits ~{steps:,} steps.')
    return steps, ips


STEPS_SMALL, IPS_SMALL = recommend_steps('small')
STEPS_LARGE, IPS_LARGE = recommend_steps('large')

print(f'\n[SUMMARY]')
print(f'  small : {IPS_SMALL:.3f} it/s -> {STEPS_SMALL:,} steps in ~{PER_CONFIG_H:.2f}h')
print(f'  large : {IPS_LARGE:.3f} it/s -> {STEPS_LARGE:,} steps in ~{PER_CONFIG_H:.2f}h')
print(f'  total wall-clock plan: ~{2*PER_CONFIG_H + 2*MARGIN_H:.1f}h (target {TOTAL_HOURS:.0f}h)')
print(f'\nThese STEPS_SMALL / STEPS_LARGE values feed directly into the training cells below.')

## 5. Train -- `small` config (the like-for-like capacity control)

Winner-equivalent capacity (hidden 112 / 6 layers, batch 512), isolating the 5 architectural changes from any capacity difference. Step count and time budget come from the calibration cell above, not a static guess.

**If this disconnects partway through, just re-run this exact cell.** It will print `[RESUME] .../checkpoint_latest.pt @ step N` and continue from there -- do not change anything (no need to re-run the calibration cell either; `STEPS_SMALL` is already set).

In [ ]:
import os
os.environ.update({
    'KMP_DUPLICATE_LIB_OK': 'TRUE',
    'PYTHONIOENCODING': 'utf-8',
    'CONFIG': 'small',
    'TRAIN_CSV': 'data/siimpl_rot/siimpl_train_v2.csv',
    'EVAL_CSV': 'data/siimpl_rot/siimpl_eval_v2.csv',
    'RESULTS_DIR': f'{RESULTS_ROOT}/small',
    'RESUME': '1',
    'TARGET_STEPS': str(STEPS_SMALL),
    'TIME_BUDGET_HOURS': str(PER_CONFIG_H),
})
%cd /content/sbi-srim
!python smearing_resolution/architecture_experiments/a100_final/train.py

## 6. Train -- `large` config (the capacity test)

~7x the winner's compute per sample (hidden 224 / 10 layers, batch 384). Run this after `small` finishes, or independently on a second A100 if you have one (same `RESULTS_ROOT`, different `RESULTS_DIR` subfolder, no conflict). Same resume behavior as above.

In [ ]:
import os
os.environ.update({
    'KMP_DUPLICATE_LIB_OK': 'TRUE',
    'PYTHONIOENCODING': 'utf-8',
    'CONFIG': 'large',
    'TRAIN_CSV': 'data/siimpl_rot/siimpl_train_v2.csv',
    'EVAL_CSV': 'data/siimpl_rot/siimpl_eval_v2.csv',
    'RESULTS_DIR': f'{RESULTS_ROOT}/large',
    'RESUME': '1',
    'TARGET_STEPS': str(STEPS_LARGE),
    'TIME_BUDGET_HOURS': str(PER_CONFIG_H),
})
%cd /content/sbi-srim
!python smearing_resolution/architecture_experiments/a100_final/train.py

## 7. Evaluate a checkpoint

Safe to run any time, including mid-run against `checkpoint_latest.pt` if you want an early read. Reports the full 9-sigma x 3-energy grid: model axis error, model head-tail %, the classical PCA baseline (same convention as the reference sweep, directly comparable), and angular-coverage ECE. **Both `CKPT` (input) and `RESULTS_DIR` (so `OUT_CSV`, which defaults to `RESULTS_DIR/full_sweep_9tier.csv`, lands there too) are explicitly pointed at Drive** -- without setting `RESULTS_DIR` here, eval.py falls back to a local-repo-relative path that lives only on the ephemeral Colab disk, which defeats the point.

In [ ]:
import os
CONFIG_TO_EVAL = 'small'  # or 'large'
CKPT_NAME = 'checkpoint_final.pt'  # or 'checkpoint_latest.pt' for a mid-run read

os.environ.update({
    'CONFIG': CONFIG_TO_EVAL,
    'RESULTS_DIR': f'{RESULTS_ROOT}/{CONFIG_TO_EVAL}',
    'CKPT': f'{RESULTS_ROOT}/{CONFIG_TO_EVAL}/{CKPT_NAME}',
    'TRAIN_CSV': 'data/siimpl_rot/siimpl_train_v2.csv',
    'EVAL_CSV': 'data/siimpl_rot/siimpl_eval_v2.csv',
    'N_PER_BIN': '1000',
})
%cd /content/sbi-srim
!python smearing_resolution/architecture_experiments/a100_final/eval.py
print(f"\nOutput CSV: {os.environ['RESULTS_DIR']}/full_sweep_9tier.csv  (on Drive)")

## Notes / troubleshooting

- **Everything under `RESULTS_ROOT` in Drive is the durable record of this run** -- `checkpoint_latest.pt` (rolling), `checkpoint_final.pt` (written at completion), `train_log.json` (loss/eval history), and `eval.py`'s `full_sweep_9tier.csv`. Nothing important should live only on the ephemeral Colab disk -- every cell that writes results explicitly sets `RESULTS_DIR` to the Drive path for exactly this reason; if a future edit adds a new output-producing cell, double check it sets `RESULTS_DIR` (or its own explicit output path) to a `RESULTS_ROOT`-based path rather than relying on `config.py`'s local-relative default.
- **`[LOADER-WAIT] N%`** appears in the training log periodically. If it stays above a few percent in the first several minutes, the CPU data pipeline is bottlenecking the GPU -- raise `N_WORKERS` (env var) above its default before continuing a long run.
- **Disconnected mid-run?** Reconnect, re-run cells 2-3 (clone + mount + stage -- cheap, idempotent), then re-run the *same* training cell (5 or 6) you were on -- `STEPS_SMALL`/`STEPS_LARGE`/`PER_CONFIG_H` need to exist in the kernel, so re-run cell 4 (calibration) too if you restarted the whole runtime, rather than editing in a guessed number. It resumes from the last Drive checkpoint automatically. To just re-run eval (cell 7), cells 2-3 are enough -- cell 4 is not needed.
- **Colab session length limits** may end a run before 30h elapses even without a real crash. This is expected, not a failure -- the resume behavior above is exactly what makes that a non-issue rather than lost work.
- Full architecture documentation, math derivations, and validation results are in `smearing_resolution/architecture_experiments/a100_final/README.md` in the cloned repo.